# Agent 上下文工程

---

`Agent 上下文工程`（Agent Context Engineering）是系统化设计和优化传递给 AI Agent 的上下文信息的技术实践，它的目标是让 Agent 在有限的上下文窗口内获得最有效的信息，从而提升任务执行的准确性、效率和可靠性。

AI Agent 不像传统程序那样有固定的执行逻辑，它的行为完全取决于接收到的上下文——包括系统提示、工具描述、历史对话和外部检索到的信息。

上下文工程的核心在于回答一个问题：**在这轮调用中，Agent 最需要知道什么？**

> 上下文工程不是一次性的提示词编写，而是贯穿 Agent 系统开发、测试和运维全生命周期的持续优化过程。

### 核心要素

Agent 上下文由以下几个层次构成：

| 层次                       | 内容               | 特点              |
| ------------------------ | ---------------- | --------------- |
| 系统提示（System Prompt）      | 角色定义、行为规则、输出格式要求 | 每次调用都会携带，相对稳定   |
| 工具定义（Tool Definitions）   | 可用工具的名称、参数和功能描述  | 占用大量 token，需要精简 |
| 历史记录（History）            | 当前会话的对话和工具调用记录   | 持续增长，需要截断或摘要策略  |
| 检索上下文（Retrieved Context） | 从外部知识库或代码库中检索的内容 | 按需注入，需要相关性排序    |
| 用户输入（User Input）         | 用户当前的指令或问题       | 不可控，但可以通过澄清来优化  |

## 核心实践

### 上下文预算管理

将上下文窗口视为一个有限的「预算」，合理分配给不同类型的上下文。

不是所有信息都值得占据上下文空间，每一条上下文都应该有明确的 ROI。

上下文预算分配建议（以 200K token 窗口为例）：

系统提示　　██████████　10%　（20K token）
工具定义　　████████████████　20%　（40K token）
检索上下文　████████████████████　25%　（50K token）
历史记录　　████████████████████████　30%　（60K token）
用户输入　　████████　5%~10%　（10-20K token）
预留缓冲　　████　5%　（10K token）

实际比例应根据 Agent 的具体任务类型动态调整。

如果 Agent 主要做代码生成，检索上下文的比例可以调高。

如果 Agent 是多轮对话助手，历史记录的预算应优先保证。

### 系统提示的工程化

系统提示是 Agent 行为的基石，需要遵循结构化编写原则。

**编写原则：**

| 原则    | 说明                    | 示例               |
| ----- | --------------------- | ---------------- |
| 分层组织  | 将指令按角色、规则、流程、格式分块     | 使用 Markdown 标题分隔 |
| 正面表述  | 告诉 Agent 应该怎么做，而非不怎么做 | 「保持回答简洁」优于「不要啰嗦」 |
| 提供示例  | 用 few-shot 示例代替长篇描述   | 输出格式示例比文字描述更高效   |
| 优先级明确 | 当规则冲突时，Agent 应遵循哪条    | 「安全规则优先于效率要求」    |
| 去除冗余  | 删除从未被触发的规则和重复说明       | 定期审查并精简系统提示      |

以下是一个结构化系统提示的示例骨架：

## 角色
你是一名 Python 代码审查助手。

## 核心规则
- 每个问题附上修复建议
- 按严重程度排序输出
- 不确定时标注「待确认」

## 工作流程
1. 分析代码变更
2. 按严重程度分类问题
3. 逐一输出问题与建议

## 输出格式
**问题**：[问题描述]
**严重程度**：[高/中/低]
**建议**：[修复建议]

### 工具描述的优化

工具描述往往占据上下文的最大比重，但很多工具从未被实际调用。

优化策略包括：

| 策略   | 方法               | 效果                 |
| ---- | ---------------- | ------------------ |
| 工具精简 | 移除与当前任务无关的工具定义   | 减少 30%~50% 的工具上下文 |
| 描述精炼 | 每个工具只用一句话说明用途和参数 | 提升模型对工具的理解准确度      |
| 参数约束 | 在描述中明确参数的使用场景和限制 | 减少工具调用中的参数错误       |
| 分组注册 | 按任务阶段动态注册不同的工具集  | 避免工具过多导致的「选择困难」    |

> 工具定义不为零即是负担。每个保留的工具都需要在描述质量和 token 成本之间做权衡。

### 历史记录的压缩策略

多轮对话中，历史记录快速增长并占据上下文。

需要选择合适的压缩策略来平衡上下文完整性和窗口限制。

常见历史压缩策略对比：

策略　　　　　方法　　　　　　　　　　　　适用场景
─────────────────────────────────────────────────
滑动窗口　　　保留最近 N 轮完整记录　　　　短对话、实时交互
阶梯摘要　　　每轮做一次增量摘要　　　　　长对话、客服场景
分层摘要　　　近期保留完整、远期做摘要　　文档生成、复杂任务
关键轮标记　　标记重要轮次，其余丢弃　　　调试、分步执行任务

分层摘要是最常用的策略：最近 3-5 轮保持完整，更早的对话用结构化摘要代替。

结构化摘要应包含原对话的关键信息：用户目标是什么，Agent 做了什么，产生了什么输出，遇到了什么错误。

### 检索上下文的质量控制

当 Agent 依赖外部知识库时，检索到的上下文质量直接决定输出质量。

提高检索上下文质量的关键点：

| 环节    | 常见问题            | 改进方法             |
| ----- | --------------- | ---------------- |
| 查询改写  | 用户输入不够精确        | 让 Agent 先改写查询再检索 |
| 相关性过滤 | 检索结果中包含无关内容     | 设置相关性阈值，不足时主动提示  |
| 来源标注  | Agent 无法判断信息可信度 | 附上来源路径和更新时间      |
| 长度裁剪  | 检索结果占用过多上下文     | 按段落截取，保留最相关片段    |

## 常见模式

### 渐进式披露

不一次性把所有信息塞进上下文，而是根据执行阶段逐步披露。

初始上下文只包含系统提示和当前阶段的必要工具，当 Agent 完成当前阶段后，再注入下一阶段的工具和规则。

这种模式将上下文的「密度」控制在合理范围内，避免 Agent 被无关信息干扰。

### 上下文压缩链

当一个任务需要处理大量数据时，用链式调用拆分为多个步骤，每步只保留关键中间结果。

步骤1：读取全部日志文件 → 输出 { 错误数量，时间范围，错误类型列表 }

步骤2：基于步骤1的输出，分析 Top 3 错误 → 输出 { 根因分析，修复建议 }

步骤3：基于步骤2的输出，生成修复 PR → 输出 { PR 标题，描述，代码变更 }

每条链路的输入都是上一步精炼后的摘要，而非原始数据，使得上下文始终保持可控大小。

### 上下文水印

在上下文的关键位置放置「水印」标记，帮助监控和诊断 Agent 行为。

上下文水印示例：

```
<system-reminder>
当前使用的上下文策略版本：v2.3
检索到的文档来源：docs/api-reference/ 共 3 个文件
已消耗上下文：85,000 / 200,000 tokens
</system-reminder>
```

这些水印不会直接影响 Agent 行为，但在调试时可以快速定位上下文注入是否按预期执行。

## 评估与迭代

上下文工程的效果需要通过评估来验证，不能仅凭感觉判断。

常用的评估维度：

| 维度     | 评估方式             | 关键指标               |
| ------ | ---------------- | ------------------ |
| 任务完成率  | 在标准测试集上运行 Agent  | 成功率、失败原因分布         |
| 上下文效率  | 统计每次调用的上下文使用量    | 平均 token 消耗、上下文利用率 |
| 工具调用质量 | 分析工具调用的准确率和参数正确率 | 无效调用占比、重试次数        |
| 响应一致性  | 相同输入多次测试         | 输出稳定性、格式合规率        |

上下文工程不是一次性配置，而是需要建立「测试 → 分析 → 优化 → 再测试」的迭代循环。

## 注意事项

> 上下文窗口不是越大越好。过多的上下文会导致模型在关键信息上的注意力被稀释，反而降低输出质量。

> 盲目增加规则可能让系统提示变得臃肿且自相矛盾。每新增一条规则，都需要审查是否与已有规则冲突。

工具定义的修改可能影响已有 Agent 的行为，修改后需要在典型场景下做回归测试。

历史记录压缩策略需要根据 Agent 的任务特征来选择，一个策略不可能在所有场景中都表现最优。

上下文中的敏感信息（API 密钥、内部路径等）会在工具调用和日志中传播，需要做好脱敏处理。

## 相关概念

| 概念                 | 说明                  |
| ------------------ | ------------------- |
| Prompt Engineering | 专注于单次调用的提示词设计和优化    |
| Context Window     | 模型能够一次性处理的 token 上限 |
| Token              | 模型处理文本的最小语义单元       |
| RAG（检索增强生成）        | 通过外部检索来扩充上下文的架构模式   |
| Few-shot Prompting | 在上下文中提供示例来引导模型输出    |
| Chain-of-Thought   | 让模型在上下文中展现推理过程      |